# Persistent Cache Statistics

New in Medha **0.5.0**.

`CacheStats` has always been an in-process counter: every restart reset the hit rate
back to zero, which makes it useless for judging cache health in production.

From 0.5.0 Medha persists a `PersistedStats` snapshot **into the backend** every
`Settings.stats_persist_interval` requests, and reloads it on `start()`. Hit rate and
request counts now survive process restarts.

This notebook covers:

1. What gets persisted (and what does not)
2. A LanceDB-backed cache that survives a simulated restart
3. Reading the metrics from the CLI with `medha stats --json`
4. Tuning `stats_persist_interval`

**Requires:** `pip install "medha-archai[fastembed,lancedb]"`

In [ ]:
import asyncio
import json
import shutil
import tempfile
from pathlib import Path

import pandas as pd

from medha import Medha, Settings
from medha.backends.lancedb import LanceDBBackend
from medha.embeddings.fastembed_adapter import FastEmbedAdapter
from medha.types import PersistedStats

# LanceDB writes to disk, so the cache (and its stats) outlive the process.
DB_PATH = Path(tempfile.mkdtemp(prefix="medha_stats_demo_"))
print("LanceDB path:", DB_PATH)

embedder = FastEmbedAdapter()
print("Embedder ready:", embedder.model_name, f"({embedder.dimension} dims)")

## 1. What is persisted

`PersistedStats` is a small Pydantic model serialised as JSON — no pickle, no vectors:

| field | meaning |
|---|---|
| `total_requests` | every `search()` call |
| `total_hits` | requests served from any tier |
| `total_misses` | `no_match` results |
| `total_errors` | `error` results |
| `hits_by_strategy` | per-tier hit counts (`l1_cache`, `semantic_match`, …) |
| `last_reset_at` / `updated_at` | timestamps (timezone-aware UTC) |

It also exposes two computed properties, `hit_rate` and `miss_rate`.

**Latency percentiles are deliberately not persisted** — they are sampled per process and
would be misleading if merged across restarts. After a restart, latency stats describe
only the current process while the counters describe the whole history.

In [ ]:
s = PersistedStats(total_requests=10, total_hits=7, total_misses=2, total_errors=1)
print("hit_rate :", s.hit_rate)
print("miss_rate:", s.miss_rate)
print()
print("serialised:", s.model_dump_json(indent=2)[:220], "...")

## 2. Populate a cache and let it persist

`stats_persist_interval=5` flushes a snapshot every 5 requests. The write happens in a
background task, so it never blocks the search path — and if it fails, the search still
returns normally (persistence is best-effort by design).

In [ ]:
QA_PAIRS = [
    ("How many users are registered?",      "SELECT COUNT(*) FROM users"),
    ("List all active products",            "SELECT * FROM products WHERE active = true"),
    ("What is the total revenue?",          "SELECT SUM(amount) FROM sales"),
    ("Show the newest orders",              "SELECT * FROM orders ORDER BY created_at DESC"),
    ("Which customers are in Italy?",       "SELECT * FROM customers WHERE country = 'IT'"),
    ("Average order value",                 "SELECT AVG(total) FROM orders"),
    ("Count pending invoices",              "SELECT COUNT(*) FROM invoices WHERE status='pending'"),
    ("Top 10 products by sales",            "SELECT * FROM products ORDER BY sold DESC LIMIT 10"),
    ("Employees hired this year",           "SELECT * FROM employees WHERE year(hired_at)=year(now())"),
    ("Total refunds issued",                "SELECT SUM(amount) FROM refunds"),
]

settings = Settings(
    backend_type="lancedb",
    lancedb_uri=str(DB_PATH),
    stats_persist_interval=5,   # flush a snapshot every 5 requests
    score_threshold_semantic=0.85,
)

async with Medha("stats_demo", embedder=embedder, backend=LanceDBBackend(settings), settings=settings) as m:
    for question, query in QA_PAIRS:
        await m.store(question, query)

    # 20 searches: 10 that should hit, 10 that should not
    for question, _ in QA_PAIRS:
        await m.search(question)
    for i in range(10):
        await m.search(f"an unrelated question about topic number {i}")

    # Drain the in-flight background writes before we leave the block.
    if m._stats_persist_tasks:
        await asyncio.gather(*m._stats_persist_tasks)

    live = await m.stats()
    print(f"in-process : {live.total_requests} requests, hit rate {live.hit_rate:.1%}")

## 3. Simulate a restart

A brand-new `Medha` object, pointing at the same LanceDB directory — the equivalent of
restarting the service. `start()` loads the snapshot before serving any traffic.

In [ ]:
restart_settings = Settings(
    backend_type="lancedb",
    lancedb_uri=str(DB_PATH),
    stats_persist_interval=5,
    score_threshold_semantic=0.85,
)

async with Medha(
    "stats_demo", embedder=embedder, backend=LanceDBBackend(restart_settings), settings=restart_settings
) as m2:
    restored = await m2.stats()
    print(f"after restart : {restored.total_requests} requests, hit rate {restored.hit_rate:.1%}")
    print()
    print("per-strategy hits:")
    for strategy, count in sorted(restored.by_strategy.items()):
        if count.count:
            print(f"  {strategy:16s} {count.count}")

Without persistence the counters above would read `0 requests, 0.0%`.

Note the number may be slightly lower than 20: with `stats_persist_interval=5` the last
snapshot lands on request 20, but any requests after the final flush are lost. That is the
accuracy/write-frequency trade-off covered in section 5.

## 4. Reading the metrics from the CLI

`medha stats` reports the persisted snapshot when one exists. This is the main reason the
feature exists: an operator can check cache health without instrumenting the application.

```bash
export MEDHA_BACKEND_TYPE=lancedb
export MEDHA_LANCEDB_URI=/path/to/db
export MEDHA_EMBEDDER_TYPE=fastembed

medha stats --collection stats_demo
medha stats --collection stats_demo --json
```

Human output gains three blocks:

```
Collection : stats_demo
Backend    : LanceDBBackend (lancedb)
Entries    : 10 (main)  0 (templates)
Requests   : 20
Hit rate   : 50.0%
By strategy:
  L1       : 10
  Template : 0
  Exact    : 0
  Semantic : 0
  Fuzzy    : 0
```

(The hits land in `L1` because `store()` warms the L1 cache, so re-asking the same
question is served before the vector tiers are ever consulted.)

and `--json` gains `total_requests`, `hit_rate`, and `hits_by_strategy`
(all `null` when nothing has been persisted yet).

In [ ]:
# The same data the CLI prints, read straight from the backend.
backend = LanceDBBackend(restart_settings)
await backend.connect()
snapshot = await backend.load_stats("stats_demo")
await backend.close()

print(json.dumps({
    "total_requests": snapshot.total_requests,
    "hit_rate": round(snapshot.hit_rate, 3),
    "hits_by_strategy": snapshot.hits_by_strategy,
}, indent=2))

## 5. Tuning `stats_persist_interval`

Every flush is one small write to the backend. The interval trades write amplification
against how many requests you can lose on an unclean shutdown.

| interval | write frequency | worst-case loss | good for |
|---|---|---|---|
| `1` | every request | 0 requests | low-traffic services, debugging |
| `100` *(default)* | every 100 requests | up to 99 | most production workloads |
| `1000` | every 1000 requests | up to 999 | high-traffic, write-sensitive backends |

Set it via `Settings(stats_persist_interval=...)` or `MEDHA_STATS_PERSIST_INTERVAL`.

Two more things worth knowing:

- Persistence only runs when `collect_stats=True` (the default). Disabling stats disables
  the writes entirely.
- A backend that does not implement `load_stats` / `save_stats` simply opts out: the two
  methods ship with a `None` / no-op default on `VectorStorageBackend`, so **custom backends
  written against 0.4.x keep working untouched**.

In [ ]:
# Cleanup
shutil.rmtree(DB_PATH, ignore_errors=True)
print("removed", DB_PATH)